# 🧠 Notebook 05 — CNN Baseline Training

---

## Project Context

| Field | Detail |
|---|---|
| **Competition** | SATRIA DATA 2026 — Big Data Challenge |
| **Experiment** | Baseline CNN — Experiment 01 |
| **Stage** | Modeling (Step 3 of 7) |
| **Primary Metric** | Macro-averaged F1 Score |

---

## Arsitektur & Hyperparameter

Berdasarkan `configs/baseline.yaml`, kita akan melatih model **Custom CNN** dengan 4 Convolutional Block.
- **Optimizer**: AdamW (LR=0.001)
- **Scheduler**: ReduceLROnPlateau
- **Loss Function**: CrossEntropyLoss
- **Early Stopping**: Berdasarkan `val_macro_f1`

> **Catatan:** Notebook ini menggunakan modul `Trainer` dari `src/training/trainer.py` untuk menjaga *Clean Architecture*.

In [ ]:
# ============================================================
# CELL 1 — IMPORTS & SETUP
# ============================================================
import sys
import logging
from pathlib import Path
import yaml

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.seed import set_seed
from src.preprocessing.splitter import collect_dataset, stratified_split
from src.preprocessing.dataloader import build_dataloaders
from src.models.cnn.baseline import build_baseline_cnn
from src.training.trainer import Trainer

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(message)s', datefmt='%H:%M:%S')
logger = logging.getLogger(__name__)

with open(PROJECT_ROOT / 'configs' / 'baseline.yaml', 'r') as f:
    config = yaml.safe_load(f)

SEED = config['experiment']['seed']
set_seed(SEED)

---
## Step 1 — Siapkan DataLoaders
Kita membangun ulang loader di sini dengan `batch_size=32`.

In [ ]:
# ============================================================
# CELL 2 — DATALOADERS
# ============================================================
data_root = PROJECT_ROOT / config['data']['raw_dir']
train_dir = data_root / config['data']['train_subdir']

df_all = collect_dataset(train_dir)
df_train, df_val = stratified_split(
    df_all, 
    train_ratio=config['split']['train_ratio'], 
    val_ratio=config['split']['val_ratio'], 
    seed=SEED
)

train_loader, val_loader = build_dataloaders(
    df_train=df_train,
    df_val=df_val,
    config=config,
    use_weighted_sampler=True
)

---
## Step 2 — Inisialisasi Model
Membangun `BaselineCNN` dari scratch tanpa pre-trained weights.

In [ ]:
# ============================================================
# CELL 3 — MODEL INITIALIZATION
# ============================================================
num_classes = len(config['data']['class_names'])
model = build_baseline_cnn(num_classes=num_classes)

print(f"Model Architecture:\n{model}")

---
## Step 3 — Training Loop
Memanggil class `Trainer` yang membungkus AdamW, ReduceLROnPlateau, EarlyStopping, dan logging.

In [ ]:
# ============================================================
# CELL 4 — TRAIN
# ============================================================
trainer = Trainer(
    model=model,
    config=config,
    project_root=PROJECT_ROOT
)

# Training akan mencetak metrik per epoch
history = trainer.train(train_loader, val_loader)
logger.info("Proses training selesai!")